# Deep Learning 基礎講座　最終課題: 脳波分類

## 概要
被験者が画像を見ているときの脳波から，その画像がどのカテゴリに属するかを分類するタスク．
- サンプル数: 訓練 118,800 サンプル，検証 59,400 サンプル，テスト 59,400 サンプル
- クラス数: 5
- 入力: 脳波データ（チャンネル数 x 系列長）
- 出力: 対応する画像のクラス
- 評価指標: Top-1 accuracy

### 元データセット ([Gifford2022 EEG dataset](https://osf.io/3jk45/)) との違い

- 本コンペでは難易度調整の目的で元データセットにいくつかの改変を加えています．

1. 訓練セットのみの使用
  - 元データセットでは訓練データに存在しなかったクラスの画像を見ているときの脳波においてテストが行われますが，これは難易度が非常に高くなります．
  - 本コンペでは元データセットの訓練セットを再分割し，訓練時に存在した画像に対応する別の脳波において検証・テストを行います．

2. クラス数の減少
  - 元データセット（の訓練セット）では16,540枚の画像に対し，1,654のクラスが存在します．
    - e.g. `aardvark`, `alligator`, `almond`, ...
  - 本コンペでは1,654のクラスを，`animal`, `food`, `clothing`, `tool`, `vehicle`の5つにまとめています．
    - e.g. `aardvark -> animal`, `alligator -> animal`, `almond -> food`, ...

### 考えられる工夫の例

- 音声モデルの導入
  - 脳波と同じ波である音声を扱うアーキテクチャを用いることが有効であると知られています．
  - 例）Conformer [[Gulati+ 2020](https://arxiv.org/abs/2005.08100)]
- 画像データを用いた事前学習
  - 本コンペのタスクは脳波のクラス分類ですが，配布してある画像データを脳波エンコーダの事前学習に用いることを許可します．
  - 例）CLIP [Radford+ 2021]
  - 画像を用いる場合は[こちら](https://osf.io/download/3v527/)からダウンロードしてください．
- 過学習を防ぐ正則化やドロップアウト


## 修了要件を満たす条件
- ベースラインモデルのbest test accuracyは38.8%となります．**これを超えた提出のみ，修了要件として認めます**．
- ベースラインから改善を加えることで，55%までは性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

## 1.準備

In [1]:
# omnicampus 実行用
!pip install ipywidgets


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ライブラリのインポートとシード固定
import os, sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from einops.layers.torch import Rearrange
from einops import repeat
from glob import glob
from termcolor import cprint
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cuda


# For Colab

In [ ]:
# ドライブのマウント（Colabの場合）
from google.colab import drive
drive.mount('/content/drive')

# For Local

In [2]:
# Set the working directory
import os
import numpy as np
import pandas as pd

#work_dir = os.path.dirname(os.path.dirname(os.getcwd())) 
work_dir = os.path.dirname(os.getcwd())

print(f"Current working directory: {work_dir}")

Current working directory: c:\Users\dysk-\Desktop\Current task\EEG compe


In [3]:
# ワーキングディレクトリを作成し移動．ノートブックを配置したディレクトリに適宜書き換え
#WORK_DIR = "/content/drive/MyDrive/weblab/DLBasics2025/Competition"
WORK_DIR = os.path.join(work_dir)
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

c:\Users\dysk-\Desktop\Current task\EEG compe


## 2.データセット

ノートブックと同じディレクトリに`data/`が存在することを確認してください．

In [4]:
import numpy as np
import torch
from torch.utils.data import Dataset

class ThingsEEGDataset(Dataset):
    def __init__(self, split: str, use_vit: bool = True):
        assert split in ["train", "val", "test"]
        self.split = split
        self.use_vit = use_vit

        self.X = np.load(f"data/{split}/eeg.npy").astype(np.float32)

        # trial-wise z-score
        self.X = (self.X - self.X.mean(axis=-1, keepdims=True)) / (
            self.X.std(axis=-1, keepdims=True) + 1e-6
        )

        self.X = np.clip(self.X, -5, 5)

        self.subject = np.load(f"data/{split}/subject_idxs.npy").astype(np.int64)
        self.subject = self.subject - 1

        if split != "test":
            self.y = np.load(f"data/{split}/labels.npy").astype(np.int64)
        else:
            self.y = None

        if use_vit and split != "test":
            self.vit = np.load(f"data/{split}/vit_features.npy").astype(np.float32)

            # ViT特徴は方向情報を使いたいのでL2 normalize
            self.vit = self.vit / (np.linalg.norm(self.vit, axis=1, keepdims=True) + 1e-6)
        else:
            self.vit = None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        subject = torch.tensor(self.subject[idx], dtype=torch.long)

        if self.split == "test":
            return x, subject

        y = torch.tensor(self.y[idx], dtype=torch.long)

        if self.use_vit:
            vit = torch.tensor(self.vit[idx], dtype=torch.float32)
            return x, subject, y, vit

        return x, subject, y

# 2.5 Load Config file

In [27]:
del run_dir

In [5]:


from pathlib import Path
from datetime import datetime
import json
import shutil

# ===== 読み込むconfigを指定 =====
#CONFIG_PATH =  Path("configs/baseline.json")
#CONFIG_PATH =  Path("configs/clip_m5_5.json")
#CONFIG_PATH =  Path("configs/baseline_zscore_clip.json")
#CONFIG_PATH =  Path("configs/eegnet_zscore_clip.json")
#CONFIG_PATH =  Path("configs/eegnet_zscore_clip_SubjectEmbedding.json")
CONFIG_PATH =  Path("configs/b_baseline_eeg_to_vit_mse_cos.json")


print(f"Loading config from: {CONFIG_PATH}")
#CONFIG_PATH = work_dir + CONFIG_PATH

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

# ===== configから変数に反映 =====
RUN_NAME = config["run_name"]
seed = config["seed"]
lr = config["lr"]
batch_size = config["batch_size"]
epochs = config["epochs"]
model_name = config["model_name"]
optimizer_name = config["optimizer"]
scheduler_name = config["scheduler"]

# ===== 保存先作成 =====
if "run_dir" not in globals():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    run_dir = Path("outputs") / f"{timestamp}_{RUN_NAME}"
    run_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy(CONFIG_PATH, run_dir / "config.json")

print(f"Run directory: {run_dir}")

Loading config from: configs\b_baseline_eeg_to_vit_mse_cos.json
Run directory: outputs\20260610_0249_b_baseline_eeg_to_vit_mse_cos


# Load image_features data

In [6]:
from pathlib import Path
import numpy as np

feature_path = work_dir + "/data/features/vit_image_features.npy"
path_txt = work_dir + "/data/features/vit_image_paths.txt"
print(feature_path)


features = np.load(feature_path)

with open(path_txt) as f:
    feature_paths = [p.strip() for p in f.readlines()]

print(features.shape)
print(len(feature_paths))
print(feature_paths[0])

c:\Users\dysk-\Desktop\Current task\EEG compe/data/features/vit_image_features.npy
(5940, 768)
5940
00001_aardvark/aardvark_01b.jpg


In [7]:
# path -> feature の辞書
feature_dict = {
    p: feat
    for p, feat in zip(feature_paths, features)
}

def make_trial_image_features(split):
    path_file = work_dir + f"/data/{split}/image_paths.txt"

    with open(path_file) as f:
        trial_paths = [p.strip() for p in f.readlines()]

    trial_features = np.stack([
        feature_dict[p]
        for p in trial_paths
    ])

    return trial_features

train_img_feats = make_trial_image_features("train")
val_img_feats = make_trial_image_features("val")

print(train_img_feats.shape)
print(val_img_feats.shape)

(118800, 768)
(59400, 768)


In [30]:
np.save(work_dir + "/data/train/vit_features.npy", train_img_feats)
np.save(work_dir + "/data/val/vit_features.npy", val_img_feats)

## 3.ベースラインモデル

In [8]:
def mse_cos_loss(pred, target, alpha=0.5):
    pred = F.normalize(pred, dim=1)
    target = F.normalize(target, dim=1)

    mse = F.mse_loss(pred, target)
    cos = 1.0 - F.cosine_similarity(pred, target, dim=1).mean()

    loss = alpha * mse + (1.0 - alpha) * cos
    return loss


def vit_regression_loss(pred, target, alpha=0.5):
    pred = F.normalize(pred, dim=1)
    target = F.normalize(target, dim=1)

    mse = F.mse_loss(pred, target)
    cos = 1.0 - F.cosine_similarity(pred, target, dim=1).mean()

    loss = alpha * mse + (1.0 - alpha) * cos
    return loss, mse.detach(), cos.detach()


class ConvBlock(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        kernel_size: int = 3,
        p_drop: float = 0.1,
    ) -> None:
        super().__init__()

        self.in_dim = in_dim
        self.out_dim = out_dim

        self.conv0 = nn.Conv1d(in_dim, out_dim, kernel_size, padding="same")
        self.conv1 = nn.Conv1d(out_dim, out_dim, kernel_size, padding="same")
        # self.conv2 = nn.Conv1d(out_dim, out_dim, kernel_size) # , padding="same")

        self.batchnorm0 = nn.BatchNorm1d(num_features=out_dim)
        self.batchnorm1 = nn.BatchNorm1d(num_features=out_dim)

        self.dropout = nn.Dropout(p_drop)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        if self.in_dim == self.out_dim:
            X = self.conv0(X) + X  # skip connection
        else:
            X = self.conv0(X)

        X = F.gelu(self.batchnorm0(X))

        X = self.conv1(X) + X  # skip connection
        X = F.gelu(self.batchnorm1(X))

        # X = self.conv2(X)
        # X = F.glu(X, dim=-2)

        return self.dropout(X)


class BasicConvClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        seq_len: int,
        in_channels: int,
        hid_dim: int = 128
    ) -> None:
        super().__init__()

        self.blocks = nn.Sequential(
            ConvBlock(in_channels, hid_dim),
            ConvBlock(hid_dim, hid_dim),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            Rearrange("b d 1 -> b d"),
            nn.Linear(hid_dim, num_classes),
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """_summary_
        Args:
            X ( b, c, t ): _description_
        Returns:
            X ( b, num_classes ): _description_
        """
        X = self.blocks(X)

        return self.head(X)
    


class EEGNetClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        num_channels: int,
        seq_len: int,
        F1: int = 32,
        D: int = 2,
        F2: int = 64,
        dropout: float = 0.5,
        subject_emb_dim: int = 16,
        num_subjects: int = 10,
    ):
        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 15), padding=(0, 7), bias=False),
            nn.BatchNorm2d(F1),
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 15), padding=(0, 7),
                      groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.subject_embedding = nn.Embedding(num_subjects, subject_emb_dim)

        with torch.no_grad():
            dummy = torch.zeros(1, num_channels, seq_len)
            feat = self._forward_features(dummy)
            feat_dim = feat.shape[1]

        self.classifier = nn.Linear(feat_dim + subject_emb_dim, num_classes)

    def _forward_features(self, x):
        x = x.unsqueeze(1)  # (batch, 1, channels, time)
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.separable(x)
        x = x.flatten(start_dim=1)
        return x

    def forward(self, x, subject_idxs):
        x = self._forward_features(x)
        subject_emb = self.subject_embedding(subject_idxs)
        x = torch.cat([x, subject_emb], dim=1)
        return self.classifier(x)


import torch
import torch.nn as nn
import torch.nn.functional as F

class EEGNetEncoder(nn.Module):
    def __init__(self, num_channels=17, num_times=100, dropout=0.25):
        super().__init__()

        F1 = 16
        D = 2
        F2 = F1 * D

        self.net = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 25), padding=(0, 12), bias=False),
            nn.BatchNorm2d(F1),

            nn.Conv2d(
                F1,
                F1 * D,
                kernel_size=(num_channels, 1),
                groups=F1,
                bias=False
            ),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),

            nn.Conv2d(
                F1 * D,
                F1 * D,
                kernel_size=(1, 15),
                padding=(0, 7),
                groups=F1 * D,
                bias=False
            ),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_times)
            out = self.net(dummy)
            self.out_dim = out.flatten(1).shape[1]

    def forward(self, x):
        x = x.unsqueeze(1)
        h = self.net(x)
        h = h.flatten(1)
        return h




import torch
import torch.nn as nn
import torch.nn.functional as F


class EEGNetEncoder(nn.Module):
    def __init__(self, num_channels=17, num_times=100, dropout=0.25):
        super().__init__()

        F1 = 16
        D = 2
        F2 = F1 * D

        self.net = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 25), padding=(0, 12), bias=False),
            nn.BatchNorm2d(F1),

            nn.Conv2d(
                F1,
                F1 * D,
                kernel_size=(num_channels, 1),
                groups=F1,
                bias=False
            ),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),

            nn.Conv2d(
                F1 * D,
                F1 * D,
                kernel_size=(1, 15),
                padding=(0, 7),
                groups=F1 * D,
                bias=False
            ),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_times)
            out = self.net(dummy)
            self.out_dim = out.flatten(1).shape[1]

    def forward(self, x):
        # x: (B, 17, 100)
        x = x.unsqueeze(1)  # (B, 1, 17, 100)
        h = self.net(x)
        h = h.flatten(1)
        return h


class EEGToViTBaseline(nn.Module):
    def __init__(
        self,
        num_classes=5,
        num_subjects=10,
        subject_dim=16,
        vit_dim=768
    ):
        super().__init__()

        self.encoder = EEGNetEncoder()
        self.subject_emb = nn.Embedding(num_subjects, subject_dim)

        hidden_dim = self.encoder.out_dim + subject_dim

        # pretrain / multitask fine-tuning用
        # EEG特徴 -> ViT特徴 768次元
        self.vit_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, vit_dim),
        )

        # 5クラス分類用
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def encode(self, x, subject):
        h = self.encoder(x)
        s = self.subject_emb(subject)
        h = torch.cat([h, s], dim=1)
        return h

    def forward_vit(self, x, subject):
        h = self.encode(x, subject)
        z = self.vit_head(h)
        z = F.normalize(z, dim=1)
        return z

    def forward_cls(self, x, subject):
        h = self.encode(x, subject)
        logits = self.classifier(h)
        return logits


def deep_cca_loss(H1, H2, outdim_size=None, r=1e-3, eps=1e-6):
    """
    H1: EEG projection, shape (batch, dim)
    H2: image projection, shape (batch, dim)

    CCAの目的:
    H1とH2をそれぞれ線形変換したときのcanonical correlationの和を最大化する。

    lossとしては負の値を返す。
    つまり、より小さいほど相関が高い。
    """
    assert H1.shape == H2.shape

    m = H1.size(0)
    o = H1.size(1)

    if outdim_size is None:
        outdim_size = o

    # batch size が projection dim より小さいとかなり不安定
    if m <= o:
        raise ValueError(f"batch size must be larger than proj dim: batch={m}, dim={o}")

    H1 = H1 - H1.mean(dim=0, keepdim=True)
    H2 = H2 - H2.mean(dim=0, keepdim=True)

    Sigma12 = (H1.T @ H2) / (m - 1)
    Sigma11 = (H1.T @ H1) / (m - 1) + r * torch.eye(o, device=H1.device)
    Sigma22 = (H2.T @ H2) / (m - 1) + r * torch.eye(o, device=H2.device)

    D1, V1 = torch.linalg.eigh(Sigma11)
    D2, V2 = torch.linalg.eigh(Sigma22)

    D1 = torch.clamp(D1, min=eps)
    D2 = torch.clamp(D2, min=eps)

    Sigma11_inv_sqrt = V1 @ torch.diag(D1.rsqrt()) @ V1.T
    Sigma22_inv_sqrt = V2 @ torch.diag(D2.rsqrt()) @ V2.T

    T = Sigma11_inv_sqrt @ Sigma12 @ Sigma22_inv_sqrt

    # singular values = canonical correlations
    S = torch.linalg.svdvals(T)

    corr = S[:outdim_size].sum()

    return -corr



def mse_cos_loss(pred, target, alpha=0.5):
    pred = F.normalize(pred, dim=1)
    target = F.normalize(target, dim=1)

    mse = F.mse_loss(pred, target)
    cos = 1.0 - F.cosine_similarity(pred, target, dim=1).mean()

    loss = alpha * mse + (1.0 - alpha) * cos
    return loss, mse.detach(), cos.detach()


def pretrain_loss_deep_cca(
    pred_vit,
    vit,
    z_eeg,
    z_img,
    cca_weight=0.005,
    cca_r=1e-3
):
    loss_mc, mse, cos = mse_cos_loss(pred_vit, vit, alpha=0.5)

    loss_cca = deep_cca_loss(
        z_eeg,
        z_img,
        outdim_size=z_eeg.size(1),
        r=cca_r
    )

    loss = loss_mc + cca_weight * loss_cca

    return loss, loss_mc.detach(), loss_cca.detach(), mse, cos

In [9]:
class EEGConformerEncoder(nn.Module):
    def __init__(
        self,
        num_channels=17,
        num_times=100,
        emb_dim=96,
        depth=3,
        num_heads=4,
        mlp_ratio=2.0,
        dropout=0.35,
        max_tokens=64,
    ):
        super().__init__()

        # EEG: (B, 17, 100)
        # Convで時間方向をpatch化し、channel方向はまとめて畳み込む
        self.patch_embed = nn.Sequential(
            nn.Conv2d(
                1,
                emb_dim,
                kernel_size=(num_channels, 15),
                stride=(1, 4),
                padding=(0, 7),
                bias=False,
            ),
            nn.BatchNorm2d(emb_dim),
            nn.ELU(),
            nn.Dropout(dropout),
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, max_tokens, emb_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=int(emb_dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth,
        )

        self.norm = nn.LayerNorm(emb_dim)
        self.out_dim = emb_dim

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        # x: (B, 17, 100)
        x = x.unsqueeze(1)  # (B, 1, 17, 100)

        x = self.patch_embed(x)  # (B, emb_dim, 1, T')
        x = x.squeeze(2)         # (B, emb_dim, T')
        x = x.transpose(1, 2)    # (B, T', emb_dim)

        B, T, D = x.shape

        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, D)
        x = torch.cat([cls, x], dim=1)          # (B, 1+T, D)

        x = x + self.pos_embed[:, : x.size(1), :]

        x = self.transformer(x)
        x = self.norm(x)

        # cls tokenを使う
        h = x[:, 0]
        return h


class EEGToViTConformer(nn.Module):
    def __init__(
        self,
        num_classes=5,
        num_subjects=10,
        subject_dim=16,
        vit_dim=768,
        emb_dim=96,
        depth=3,
        num_heads=4,
        dropout=0.35,
    ):
        super().__init__()

        self.encoder = EEGConformerEncoder(
            num_channels=17,
            num_times=100,
            emb_dim=emb_dim,
            depth=depth,
            num_heads=num_heads,
            dropout=dropout,
        )

        self.subject_emb = nn.Embedding(num_subjects, subject_dim)

        hidden_dim = self.encoder.out_dim + subject_dim

        self.vit_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, vit_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def encode(self, x, subject):
        h = self.encoder(x)
        s = self.subject_emb(subject)
        h = torch.cat([h, s], dim=1)
        return h

    def forward_vit(self, x, subject):
        h = self.encode(x, subject)
        z = self.vit_head(h)
        z = F.normalize(z, dim=1)
        return z

    def forward_cls(self, x, subject):
        h = self.encode(x, subject)
        logits = self.classifier(h)
        return logits

# Set Seed

In [10]:
import random
import numpy as np
import torch

def seed_everything(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(seed)

## 4.訓練実行

In [11]:
RUN_NAME = "c_conformer_eeg_to_vit_mse_cos"

train_ds = ThingsEEGDataset("train", use_vit=True)
val_ds = ThingsEEGDataset("val", use_vit=True)

train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_ds,
    batch_size=512,
    shuffle=False,
    num_workers=0,
)

model = EEGToViTConformer(
    emb_dim=96,
    depth=3,
    num_heads=4,
    dropout=0.35,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=8e-4,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
)

best_val_loss = float("inf")

for epoch in range(30):
    model.train()

    train_loss = 0.0
    train_mse = 0.0
    train_cos = 0.0
    train_n = 0

    for x, subject, y, vit in tqdm(train_loader, desc=f"conformer pretrain {epoch+1}"):
        x = x.to(device)
        subject = subject.to(device)
        vit = vit.to(device)

        optimizer.zero_grad()

        pred_vit = model.forward_vit(x, subject)

        loss, mse, cos = vit_regression_loss(
            pred_vit,
            vit,
            alpha=0.5,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = x.size(0)
        train_loss += loss.item() * bs
        train_mse += mse.item() * bs
        train_cos += cos.item() * bs
        train_n += bs

    scheduler.step()

    train_loss /= train_n
    train_mse /= train_n
    train_cos /= train_n

    model.eval()

    val_loss = 0.0
    val_mse = 0.0
    val_cos_loss = 0.0
    val_cos_sim = 0.0
    val_n = 0

    with torch.no_grad():
        for x, subject, y, vit in val_loader:
            x = x.to(device)
            subject = subject.to(device)
            vit = vit.to(device)

            pred_vit = model.forward_vit(x, subject)

            loss, mse, cos = vit_regression_loss(
                pred_vit,
                vit,
                alpha=0.5,
            )

            cos_sim = F.cosine_similarity(
                F.normalize(pred_vit, dim=1),
                F.normalize(vit, dim=1),
                dim=1,
            ).mean()

            bs = x.size(0)
            val_loss += loss.item() * bs
            val_mse += mse.item() * bs
            val_cos_loss += cos.item() * bs
            val_cos_sim += cos_sim.item() * bs
            val_n += bs

    val_loss /= val_n
    val_mse /= val_n
    val_cos_loss /= val_n
    val_cos_sim /= val_n

    print(
        f"epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.5f} | "
        f"train_mse={train_mse:.5f} | "
        f"train_cos_loss={train_cos:.5f} | "
        f"val_loss={val_loss:.5f} | "
        f"val_mse={val_mse:.5f} | "
        f"val_cos_loss={val_cos_loss:.5f} | "
        f"val_cos_sim={val_cos_sim:.5f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "model_c_conformer_pretrained.pt")
        print("saved: model_c_conformer_pretrained.pt")

c:\Users\dysk-\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


conformer pretrain 1:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 01 | train_loss=0.42983 | train_mse=0.00223 | train_cos_loss=0.85743 | val_loss=0.42704 | val_mse=0.00222 | val_cos_loss=0.85187 | val_cos_sim=0.14813
saved: model_c_conformer_pretrained.pt


conformer pretrain 2:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 02 | train_loss=0.42735 | train_mse=0.00222 | train_cos_loss=0.85249 | val_loss=0.42698 | val_mse=0.00222 | val_cos_loss=0.85174 | val_cos_sim=0.14826
saved: model_c_conformer_pretrained.pt


...

epoch 28 | train_loss=0.41180 | train_mse=0.00214 | train_cos_loss=0.82147 | val_loss=0.41278 | val_mse=0.00214 | val_cos_loss=0.82342 | val_cos_sim=0.17658
saved: model_c_conformer_pretrained.pt


conformer pretrain 29:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 29 | train_loss=0.41184 | train_mse=0.00214 | train_cos_loss=0.82155 | val_loss=0.41281 | val_mse=0.00214 | val_cos_loss=0.82347 | val_cos_sim=0.17653


conformer pretrain 30:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 30 | train_loss=0.41184 | train_mse=0.00214 | train_cos_loss=0.82155 | val_loss=0.41285 | val_mse=0.00214 | val_cos_loss=0.82355 | val_cos_sim=0.17645


In [12]:
train_ds_ft = ThingsEEGDataset("train", use_vit=False)
val_ds_ft = ThingsEEGDataset("val", use_vit=False)

train_loader_ft = DataLoader(
    train_ds_ft,
    batch_size=256,
    shuffle=True,
    num_workers=0,
)

val_loader_ft = DataLoader(
    val_ds_ft,
    batch_size=512,
    shuffle=False,
    num_workers=0,
)

model = EEGToViTConformer(
    emb_dim=96,
    depth=3,
    num_heads=4,
    dropout=0.35,
).to(device)

model.load_state_dict(
    torch.load("model_c_conformer_pretrained.pt", map_location=device)
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(
    [
        {"params": model.encoder.parameters(), "lr": 2e-4},
        {"params": model.subject_emb.parameters(), "lr": 2e-4},
        {"params": model.classifier.parameters(), "lr": 8e-4},
    ],
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
)

best_val_acc = 0.0

for epoch in range(50):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_n = 0

    for x, subject, y in tqdm(train_loader_ft, desc=f"conformer finetune {epoch+1}"):
        x = x.to(device)
        subject = subject.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model.forward_cls(x, subject)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = x.size(0)
        train_loss += loss.item() * bs
        train_correct += (logits.argmax(dim=1) == y).sum().item()
        train_n += bs

    scheduler.step()

    train_loss /= train_n
    train_acc = train_correct / train_n

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_n = 0

    with torch.no_grad():
        for x, subject, y in val_loader_ft:
            x = x.to(device)
            subject = subject.to(device)
            y = y.to(device)

            logits = model.forward_cls(x, subject)
            loss = criterion(logits, y)

            bs = x.size(0)
            val_loss += loss.item() * bs
            val_correct += (logits.argmax(dim=1) == y).sum().item()
            val_n += bs

    val_loss /= val_n
    val_acc = val_correct / val_n

    print(
        f"epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.5f} | train_acc={train_acc:.5f} | "
        f"val_loss={val_loss:.5f} | val_acc={val_acc:.5f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "model_c_conformer_finetuned_best.pt")
        torch.save(model.state_dict(), "model_best.pt")
        print(f"saved: model_c_conformer_finetuned_best.pt | val_acc={best_val_acc:.5f}")

C:\Users\dysk-\AppData\Local\Temp\ipykernel_22696\3620931312.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("model_c_conformer_pretrained.pt", map_location=

conformer finetune 1:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 01 | train_loss=1.36712 | train_acc=0.47342 | val_loss=1.34670 | val_acc=0.48030
saved: model_c_conformer_finetuned_best.pt | val_acc=0.48030


conformer finetune 2:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 02 | train_loss=1.34912 | train_acc=0.48233 | val_loss=1.33843 | val_acc=0.48184
saved: model_c_conformer_finetuned_best.pt | val_acc=0.48184


...

epoch 48 | train_loss=1.29348 | train_acc=0.50588 | val_loss=1.31084 | val_acc=0.49572


conformer finetune 49:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 49 | train_loss=1.29174 | train_acc=0.50690 | val_loss=1.31243 | val_acc=0.49554


conformer finetune 50:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 50 | train_loss=1.29332 | train_acc=0.50658 | val_loss=1.31114 | val_acc=0.49609


## 5.評価

In [13]:
test_ds = ThingsEEGDataset("test", use_vit=False)
test_loader = DataLoader(
    test_ds,
    batch_size=512,
    shuffle=False,
    num_workers=0,
)

model = EEGToViTConformer(
    emb_dim=96,
    depth=3,
    num_heads=4,
    dropout=0.35,
).to(device)

model.load_state_dict(
    torch.load("model_c_conformer_finetuned_best.pt", map_location=device)
)

model.eval()

all_probs = []

with torch.no_grad():
    for x, subject in tqdm(test_loader, desc="predict conformer"):
        x = x.to(device)
        subject = subject.to(device)

        logits = model.forward_cls(x, subject)
        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)
y_pred = all_probs.argmax(axis=1)

np.save("submission.npy", all_probs)
np.save("probs_c_conformer.npy", all_probs)
np.save("y_pred_c_conformer.npy", y_pred)

print("submission:", all_probs.shape)
print("y_pred:", y_pred.shape)
print("row sum:", all_probs.sum(axis=1)[:5])

C:\Users\dysk-\AppData\Local\Temp\ipykernel_22696\655235376.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("model_c_conformer_finetuned_best.pt", map_locati

predict conformer:   0%|          | 0/117 [00:00<?, ?it/s]

submission: (59400, 5)
y_pred: (59400,)
row sum: [1.0000001  1.         0.99999994 1.         1.        ]


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (EEG)」から提出してください．

- `submission.npy`
- `model_last.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [14]:
from zipfile import ZipFile
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_name = run_dir / f"{timestamp}_submission.zip"

submission_path = run_dir / "submission.npy"
model_path = run_dir / "model_best.pt"
notebook_path = Path(work_dir) / "notebooks" / "DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb"

with ZipFile(zip_name, "w") as zf:
    zf.write(submission_path, arcname="submission.npy")
    zf.write(model_path, arcname="model_best.pt")
    zf.write(notebook_path, arcname="DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb")

print(f"Created: {zip_name}")

with ZipFile(zip_name, "r") as zf:
    print(zf.namelist())

Created: outputs\20260610_0249_b_baseline_eeg_to_vit_mse_cos\20260610_0308_submission.zip
['submission.npy', 'model_best.pt', 'DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb']


In [19]:
sub = np.load(f"{run_dir}/submission.npy")
print(sub.shape)
print(sub.ndim)
print(sub[:3])
print(sub.sum(axis=1)[:5])

(59400,)
1
[1 1 1]


AxisError: axis 1 is out of bounds for array of dimension 1